# Land-cover classification on Colab
Run the cells top to bottom. Set the runtime to **GPU** (Runtime > Change runtime type) before training.

In [ ]:
!git clone https://github.com/<your-username>/landcover-classification.git
%cd landcover-classification
!pip install -q rasterio geopandas earthengine-api

## 1. Export imagery from Earth Engine to Drive

In [ ]:
import ee
ee.Authenticate()
!python scripts/gee_export.py --project YOUR_GCP_PROJECT

## 2. Mount Drive and copy the exported GeoTIFFs into `data/raw/`
Files must be named `<region>_s2.tif` and `<region>_label.tif`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/landcover/*.tif data/raw/
!ls data/raw

## 3. Build patches, then train and evaluate

In [ ]:
!python -m landcover.preprocess
!python -m landcover.baseline_rf
!python -m landcover.train --model cnn --epochs 30
!python -m landcover.train --model unet --epochs 40

In [ ]:
!python -m landcover.evaluate --model cnn
!python -m landcover.evaluate --model unet

## 4. Classify a scene and view the results

In [ ]:
!python -m landcover.predict --input data/raw/<test_region>_s2.tif --model-type unet --vectorize
from IPython.display import Image
Image('outputs/unet_test_confusion_matrix.png')